<a href="https://colab.research.google.com/github/elhamod/BA305_Fall_2026/blob/main/Session%2027%20-%20Optimization%20III%20-%20Integer%20Programming/lab11_IP_Google_Adwords.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Optimization Case: Google Keyword Optimization
© 2021 Sami Karaca (huseyin.karaca@boun.edu.tr), Gerry Tsoukalas (gerryt@bu.edu)

## Objectives:

In this module, we will learn how to use Binary (Integer) Programming to solve a keyword optimization problem. We will:

*   translate a problem description into a mathematical (algebraic) model
*   embed this model into a python optimization framework
*   learn how to use binary variables to express constraints
*   choose the correct solution algorithm
*   solve and interpret the model


## Problem Description

See Slides.

## Preliminary: Import Packages

In [ ]:
# Install Pyomo optimization modeling package
!pip install pyomo
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.opt import SolverFactory

In [ ]:
# Install optimization engine (solution algorithms)
# Linear program solver https://www.gnu.org/software/glpk/
!apt-get install -y -qq glpk-utils

## Load Data and Pre-processing

In [ ]:
import numpy as np
n=4 #number of different keywords
m=8 #number of different slots per keyword

costdata=np.array([
  [6.74, 1.30, 0.60, 0.94],
  [6.73, 0.96, 0.59, 0.86],
  [2.01, 0.81, 0.57, 0.85],
  [1.71, 0.80, 0.56, 0.71],
  [1.57, 0.79, 0.52, 0.59],
  [1.43, 0.78, 0.43, 0.47],
  [1.14, 0.58, 0.41, 0.44],
  [0.96, 0.55, 0.40, 0.41],
])

clickdata=np.array([
 [1539, 370, 45, 38],
 [1099, 264, 32, 27],
 [785, 189, 23, 19],
 [561, 135, 16, 14],
 [401, 96, 12, 10],
 [286, 69, 8, 7],
 [204, 49, 6, 5],
 [146, 35, 4, 4],
])

# Calculate total costs which is component-wise multiplication
totalcosts = clickdata * costdata

## Optimization Model


In [ ]:
# Initialize model by defining a pyomo model
model = pyo.ConcreteModel()

# ------------------------------------
# Decision Variables                  |
# ------------------------------------

model.m = Set(initialize=range(m))
model.n = Set(initialize=range(n))
model.x = Var(model.m, model.n, within=Boolean)

# Shorthand notation
x = model.x

# ------------------------------------
# Obj Function                        |
# ------------------------------------

# Maximize clicks.
# For the double sum over n and m, we will use a vector index k defined as k = m * n
model.obj = pyo.Objective(expr = sum(x[k]*clickdata[k] for k in model.m*model.n),  sense = maximize)

# One possible alternative formulation that involves defining the double sum
# more epxlicitly, is as follows:
# model.obj = pyo.Objective(expr = sum( sum(x[i,j]*clickdata[i,j] for i in model.m) for j in model.n),  sense = maximize)


# ------------------------------------
# CONSTRAINTS                         |
# ------------------------------------

# Only "1 slot per keyword" constraint
model.Cslots = ConstraintList()
for j in range(n):
  model.Cslots.add(sum(x[i,j] for i in model.m) <= 1)

# Budget constraint
model.CBudget = pyo.Constraint(expr = sum(x[k]*totalcosts[k] for k in model.m*model.n) <= 550)

# ------------------------------------
# SOLVE THE MODEL                     |
# ------------------------------------

# Specify the solver engine. 
opt = SolverFactory('glpk')

results = opt.solve(model)

In [ ]:
# Better visualize the output
!pip install tabulate
from tabulate import tabulate as tb
import numpy as np

# Create table header (first row)
table = np.array([
         ['Slot', 'Keyword A','Keyword B', 'Keyword C', 'Keyword D']
])

# Fill in subsequent rows
for j in range(m):
  row = [j+1, x[j,0](),x[j,1](),x[j,2](),x[j,3]()]
  table = np.append(table,[row],axis= 0)

# Print results table
print('\n Results:')
print(str(results.solver))
print(tb(table,  tablefmt='fancy_grid'))

# Check Solution

In [ ]:
print("Optimal #of clicks obtained (this is the OFV):", value(model.obj), " clicks")
print("total cost of marketing plan:", value(model.CBudget), " out of $550 available")